# Análisis de emisiones de CO₂ en Colombia
**Daniel Carrillo y Daniela** — Actividad 4, Ciencia de Datos.

Fuente: Our World in Data, indicador de emisiones anuales de CO₂. Este primer avance se centra en descarga, inspección y limpieza.


In [ ]:
import pandas as pd
url = "https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.columns.tolist())
df.head()


## Selección de Colombia
Filtramos por código ISO COL y conservamos la serie temporal de emisiones.


In [ ]:
indicador = next(c for c in df.columns if c not in ["Entity", "Code", "Year"])
co = df.loc[df["Code"].eq("COL"), ["Entity", "Code", "Year", indicador]].copy()
co.columns = ["pais", "codigo", "anio", "co2_toneladas"]
co.head()


## Diagnóstico y limpieza
Revisamos nulos, duplicados, años repetidos y emisiones negativas. No imputamos datos faltantes sin justificación.


In [ ]:
print("Nulos:", co.isna().sum())
print("Duplicados:", co.duplicated().sum())
co["anio"] = pd.to_numeric(co["anio"], errors="coerce")
co["co2_toneladas"] = pd.to_numeric(co["co2_toneladas"], errors="coerce")
co = co.drop_duplicates().dropna(subset=["anio", "co2_toneladas"])
co = co[co["co2_toneladas"] >= 0].sort_values("anio").reset_index(drop=True)
assert not co["anio"].duplicated().any(), "Existen años repetidos"
co["anio"] = co["anio"].astype(int)
co["co2_millones_toneladas"] = co["co2_toneladas"] / 1e6
print("Años:", co.anio.min(), co.anio.max())
co.describe()


## Exportación
Ejecutar desde la raíz del proyecto para guardar los datos limpios.


In [ ]:
from pathlib import Path
Path("datos").mkdir(exist_ok=True)
co.to_csv("datos/co2_colombia_limpio.csv", index=False)
print("CSV guardado")


## Siguientes avances
1. Daniela: visualizaciones y análisis exploratorio.
2. Ambos: regresión lineal con evaluación temporal, interpretación y conclusiones.

Referencia: Ritchie y Roser, *CO₂ emissions*, Our World in Data, https://ourworldindata.org/co2-emissions.
